# Lab 3: Explain, Fix, and Optimize

A coworker of yours had been working on an analysis of players and games in the Professional Football League, using the PFL dataset. However, they have left for an unexpected leave of absence and their work is now your responsibility. As you begin looking through their work, you find lots of SQL and Python code, some of which is broken, some of which runs slow, all of which is confusing.

**Objective:** In this lab exercise, you will use Snowflake CoCo to explain what unfamiliar code is doing, fix broken code, and optimize code that is running slowly.

## Setup

First, let's get your environment set up. Run the cell below to generate the PFL dataset - a database of information about the fictional Professional Football League.

In [ ]:
from resources.setup_pfl import setup_pfl
setup_pfl()

The database `PFL_DB` has been created with the schema `STATS_AND_INFO` that holds all of the tables and views relating to the Professional Football League.

Run the cell below to set your context to this database and schema.

In [ ]:
%%sql -r Set_Context
CREATE WAREHOUSE IF NOT EXISTS COCOLABS_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE COCOLABS_WH;
USE DATABASE PFL_DB;
USE SCHEMA STATS_AND_INFO;

## Explaining Code

Code can often be confusing. Ideally, whoever wrote the code has provided helpful comments. But oftentimes this isn't the case. 

Fortunately, Snowflake CoCo is great at reading and explaining code. And in Snowsight, this can be done easily with just a click of a button.

### Explain a complex SQL statement

The SQL statement in the cell below is a bit complex. Glancing at it quickly, we can get a look at some of its elements - there's the main SELECT statement, a JOIN clause... but what does this SQL actually produce.

Highlight the full statement in the SQL cell below. Once you do, click the **Explain** button that appears below.

<div style="max-width: 700px;">

![](resources/img/explain_sql_button.png)</div>

In [ ]:
%%sql -r Complex_SQL
WITH ranked AS (
    SELECT player_name, team_name, season_year,
        completion_pct, pass_yards, pass_touchdowns, passer_rating,
        ROW_NUMBER() OVER (PARTITION BY season_year ORDER BY passer_rating DESC) AS rk
    FROM v_passing_leaders
    WHERE season_year IN (2023, 2024)
)
SELECT
    r23.player_name, r23.team_name,
    r23.completion_pct AS pct_2023, r24.completion_pct AS pct_2024,
    r23.pass_yards AS yards_2023, r24.pass_yards AS yards_2024,
    r23.passer_rating AS rating_2023, r24.passer_rating AS rating_2024,
    ROUND(r24.passer_rating - r23.passer_rating, 1) AS rating_change
FROM ranked r23
JOIN ranked r24 ON r23.player_name = r24.player_name
WHERE r23.season_year = 2023 AND r24.season_year = 2024
  AND (r23.rk <= 5 OR r24.rk <= 5)
ORDER BY r24.passer_rating DESC;

Snowflake CoCo should have produced an explanation of what the SQL statement above accomplishes. In addition, it likely produced a breakdown of different steps within the SQL to help you understand how it works.

Run the SQL cell above to see its output. Does it fit with what CoCo explained?

### Explaining segments of code

In addition to explaining a full SQL statement, CoCo can explain just a segment. This can be helpful if you want more detail about how a specific clause works or what it accomplishes.

Return to the SQL block above and highlight only the line that begins with JOIN (line 15). Then, click the **Explain** button.

What is the purpose of the join clause?

What do **r23** and **r24** stand for?

If CoCo did not already answer these questions, just ask it!

### Explain some confusing Python

Your coworker has written a number of Python functions that are a bit difficult to understand. Worst of all, there are no comments! What does the Python cell below do?

Highlight the entirety of the cell below, and click the **Explain** button.

<div style="max-width: 700px;">

![](resources/img/explain_python_button.png)</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.snowpark.context as ctx
session = ctx.get_active_session()
df = session.sql("""
    SELECT t.full_name AS team_name, d.division_name, c.conference_name,
           ts.wins, ts.losses, ts.points_scored, ts.points_allowed,
           ts.offense_rank, ts.defense_rank, ts.made_playoffs
    FROM team_season_stats ts
    JOIN teams t ON ts.team_id = t.team_id
    JOIN divisions d ON t.division_id = d.division_id
    JOIN conferences c ON d.conference_id = c.conference_id
    WHERE ts.season_id = 2024
""").to_pandas()
fig, ax = plt.subplots(figsize=(10, 8))
for conf in df['CONFERENCE_NAME'].unique():
    subset = df[df['CONFERENCE_NAME'] == conf]
    playoff = subset[subset['MADE_PLAYOFFS'] == True]
    non_playoff = subset[subset['MADE_PLAYOFFS'] == False]
    ax.scatter(non_playoff['OFFENSE_RANK'], non_playoff['DEFENSE_RANK'],
               label=f'{conf}', alpha=0.6, s=80)
    ax.scatter(playoff['OFFENSE_RANK'], playoff['DEFENSE_RANK'],
               marker='*', s=200, edgecolors='black', linewidth=1,
               label=f'{conf} (Playoff)')
for _, row in df[df['MADE_PLAYOFFS'] == True].iterrows():
    ax.annotate(row['TEAM_NAME'].split()[-1],
                (row['OFFENSE_RANK'], row['DEFENSE_RANK']),
                fontsize=8, ha='left', va='bottom')
ax.set_xlabel('Offense Rank (lower = better)')
ax.set_ylabel('Defense Rank (lower = better)')
ax.set_title('2024 PFL: Offense vs Defense Rankings')
ax.legend(loc='lower right')
ax.invert_xaxis(); ax.invert_yaxis()
plt.tight_layout(); plt.show()

Run the Python cell above. Did it do what CoCo said it would?

### Add some comments

You want to be able to understand that code next time you run across it without having to ask Snowflake CoCo. Adding some code comments would be the perfect solution (and one your coworker should have done...)

The Python code is duplicated in the cell below. When prompting CoCo, you can reference a specific cell in a notebook by using its name. The name of the cell below is **Add_Python_Comments**.

In the CoCo side panel, enter the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Add comments to the "Add_Python_Comments" cell 
 that briefly explain what each section of code does.
</code>

CoCo may take a bit longer to accomplish this. 

When it finishes, click the **Keep all** button and examine the new code comments in the cell below.

<div style="max-width: 400px;">

![](resources/img/keep_all_button.png)</div>

In [ ]:
# Imports and Snowpark session setup
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.snowpark.context as ctx
session = ctx.get_active_session()

# Query 2024 team stats joined with division and conference info
df = session.sql("""
    SELECT t.full_name AS team_name, d.division_name, c.conference_name,
           ts.wins, ts.losses, ts.points_scored, ts.points_allowed,
           ts.offense_rank, ts.defense_rank, ts.made_playoffs
    FROM team_season_stats ts
    JOIN teams t ON ts.team_id = t.team_id
    JOIN divisions d ON t.division_id = d.division_id
    JOIN conferences c ON d.conference_id = c.conference_id
    WHERE ts.season_id = 2024
""").to_pandas()

# Create scatter plot, grouping teams by conference and playoff status
fig, ax = plt.subplots(figsize=(10, 8))
for conf in df['CONFERENCE_NAME'].unique():
    subset = df[df['CONFERENCE_NAME'] == conf]
    playoff = subset[subset['MADE_PLAYOFFS'] == True]
    non_playoff = subset[subset['MADE_PLAYOFFS'] == False]
    # Non-playoff teams as circles
    ax.scatter(non_playoff['OFFENSE_RANK'], non_playoff['DEFENSE_RANK'],
               label=f'{conf}', alpha=0.6, s=80)
    # Playoff teams as stars with black border
    ax.scatter(playoff['OFFENSE_RANK'], playoff['DEFENSE_RANK'],
               marker='*', s=200, edgecolors='black', linewidth=1,
               label=f'{conf} (Playoff)')

# Label playoff teams with their team name
for _, row in df[df['MADE_PLAYOFFS'] == True].iterrows():
    ax.annotate(row['TEAM_NAME'].split()[-1],
                (row['OFFENSE_RANK'], row['DEFENSE_RANK']),
                fontsize=8, ha='left', va='bottom')

# Format axes (inverted so rank 1 = best appears at top-right)
ax.set_xlabel('Offense Rank (lower = better)')
ax.set_ylabel('Defense Rank (lower = better)')
ax.set_title('2024 PFL: Offense vs Defense Rankings')
ax.legend(loc='lower right')
ax.invert_xaxis(); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## Fix Code with Snowflake CoCo

As you continue looking through your coworker's code, you find that some of the SQL statements do not run or do not deliver what they promise. They seem to have various errors that are not easy to pinpoint.

### Fix a Broken SQL statement

The cell below has some errors. Although it will fail, run it anyway to see what happens.

The output shows that there is an error on line 2 due to an invalid identifier: `t.team_full_name` is not the name of a real column. Do you know the correct column name off the top of your head? Probably not.

The old way of fixing this would be to lookup the real column name, replace it, and try running the cell again. But let's not do things the old way.

Instead, click the **Fix error** button that appeared.

<div style="max-width: 700px;">

![](resources/img/fix_error_button.png)</div>

In [ ]:
%%sql -r Broken_SQL
SELECT
    t.FULL_NAME,
    ts.WINS, ts.LOSSES,
    CONCAT(ts.POINTS_SCORED, ' - ', ts.POINTS_ALLOWED) AS SCORE_LINE,
    ROUND(ts.WINS * 100.0 / NULLIF(ts.WINS + ts.LOSSES, 0), 1) AS WIN_RATE
FROM TEAM_SEASON_STATS ts
JOIN TEAMS t ON ts.TEAM_ID = t.TEAM_ID
WHERE ts.SEASON_ID = 2024
ORDER BY WIN_RATE DESC;

Not only did CoCo find and fix the error on line 2, it found and fixed some more! In the CoCo panel, you should see a summary of the issues that it fixed. Normally, this process would have taken a few iterations of testing and fixing.

Take a look at the SQL cell above. Each change that CoCo made is annotated using a red and green background. This Diff view shows what CoCo removed (in red) and what it added (in green.) This lets you review any changes, in case CoCo made any incorrect assumptions.

Go through and click the keep button next to each of the changes.

<div style="max-width:650px">

![](resources/img/diff_view_keep.png)</div>

### Modify a query

Run the SQL cell below.

In [ ]:
%%sql -r Incomplete_NEM_Scores
-- Show Scores for New England Minutemen (NEM) with opposing team and running total
SELECT
    g.week_number, g.game_date, g.home_score, g.away_score
FROM games g
WHERE g.home_team_id = 'NEM' OR g.away_team_id = 'NEM'
ORDER BY g.week_number;

Although this cell runs successfully, it does not deliver what it is supposed to. The comment above it claims that it will show the opposing team and a running total of the points scored by the New England Minutemen, but neither of those are included in the output.

The SQL cell has been duplicated below with the name **Enhanced_NEM_Scores**. You will modify that cell to do two things:

1. Include the name of the opposing team
1. Calculate a running total of the NEM team

This time, we will not tell you the exact prompts to type. That's up to you. Just be sure to tell Snowflake CoCo to update the **Enhanced_NEM_Scores** cell and clearly state what you want. Check the output and ask again if the results aren't quite right.

<details style="outline:1px solid #aeaeae; padding:5px; border-radius:5px;">
<summary style="font-weight:bold;cursor:pointer;">Click Here For Help</summary>

Here is a prompt that gave a successful result:

<code>
Modify the Enhanced_NEM_Scores cell to include the name of the opposing team and calculate a running total of the NEM team.
</code>

</details>


In [ ]:
%%sql -r Enhanced_NEM_Scores
-- Show Scores for New England Minutemen (NEM) with opposing team and running total
SELECT
    g.week_number,
    g.game_date,
    opp.FULL_NAME AS opponent,
    g.home_score,
    g.away_score,
    CASE WHEN g.home_team_id = 'NEM' THEN g.home_score ELSE g.away_score END AS nem_score,
    CASE WHEN g.home_team_id = 'NEM' THEN g.away_score ELSE g.home_score END AS opp_score,
    SUM(CASE WHEN g.home_team_id = 'NEM' THEN g.home_score ELSE g.away_score END)
        OVER (ORDER BY g.week_number) AS nem_running_total
FROM games g
JOIN teams opp
  ON opp.team_id = CASE WHEN g.home_team_id = 'NEM' THEN g.away_team_id ELSE g.home_team_id END
WHERE g.home_team_id = 'NEM' OR g.away_team_id = 'NEM'
ORDER BY g.week_number;

### Optimize a slow query

While testing one of your coworker's queries, you get the suspicion that it is taking longer than it should.

Run the SQL query in the cell below, and make a note of how long it took to complete.

In [ ]:
%%sql -r Passing_Stats_Slow
SELECT
    p.first_name || ' ' || p.last_name AS player_name,
    t.full_name AS team_name, pss.pass_yards, pss.season_id,
    (SELECT MAX(pss2.pass_yards) FROM player_season_stats pss2
     WHERE pss2.team_id = pss.team_id AND pss2.season_id = pss.season_id) AS team_max_pass_yards,
    (SELECT AVG(pss3.pass_yards) FROM player_season_stats pss3
     WHERE pss3.season_id = pss.season_id AND pss3.pass_yards > 0) AS league_avg_pass_yards
FROM player_season_stats pss
JOIN players p ON pss.player_id = p.player_id
JOIN teams t ON pss.team_id = t.team_id
WHERE pss.pass_yards > 0
ORDER BY pss.pass_yards DESC;

The same SQL statement has been duplicated below. Let's optimize its performance. But how?

Highlight everything in the cell below and click the **Explain** button.

Although results from CoCo may vary, it is likely that it noticed some performance issues and reported them to you. It may even suggest a prompt that you can give in order to fix the issues. If so, click the suggested prompt and run it.

<div style="max-width:600px;">

![](resources/img/Performance_improvement.png)</div>

If CoCo did not make these suggestions, try the following prompts, as necessary:

- How can I improve performance in the Passing_Stats_Optimized cell?
- Rewrite the Performance_Stats_Optimized cell to use window functions instead of correlated subqueries to improve performance.

In [ ]:
%%sql -r Passing_Stats_Optimized
SELECT
    p.first_name || ' ' || p.last_name AS player_name,
    t.full_name AS team_name,
    pss.pass_yards,
    pss.season_id,
    MAX(pss.pass_yards) OVER (PARTITION BY pss.team_id, pss.season_id) AS team_max_pass_yards,
    AVG(pss.pass_yards) OVER (PARTITION BY pss.season_id) AS league_avg_pass_yards
FROM player_season_stats pss
JOIN players p ON pss.player_id = p.player_id
JOIN teams t ON pss.team_id = t.team_id
WHERE pss.pass_yards > 0
ORDER BY pss.pass_yards DESC;

Run the optimized query. Did the performance improve?

## Key Takeaways

In this lab, you explored how Snowflake CoCo can be used to:

- Use the Explain feature to understand confusing code
- Modify code to add comments
- Fix broken code
- Confirm or reject changes using Diff view
- Optimize queries for better performance